# Saffron-v1 — training on SageMaker Studio Lab

Free T4 GPU, ~4h GPU sessions. Training **auto-resumes** from `results/ckpt.pt`, so a session timeout is harmless — just re-run the train cell in a new session.

**Runtime tips**
- Use the **CPU runtime** for *Setup* + *Prepare data* (don't burn GPU hours on data prep).
- Switch to the **GPU runtime** for *Train*.

**Data flow:** local → GitHub (code) → Studio Lab (`git clone`/`pull`) → train on T4 → bring weights back (download `results/saffron.pt`, or push to Hugging Face later).

## 0. Get the code (run once per Studio Lab project)
In a **Terminal** (File → New → Terminal):
```bash
git clone https://github.com/shishodiaabhilash/saffron-v1.git
cd saffron-v1
bash scripts/setup_studiolab.sh   # creates the 'saffron' conda env with CUDA torch
```
Then set this notebook's kernel to the **saffron** env.

In [ ]:
import torch
print('torch', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))

## 1. Prepare data (CPU runtime is fine)
Streams + tokenizes ~400M tokens (TinyStories → Wikipedia → FineWeb-Edu) into `data/train.bin` / `data/val.bin`. Run once; the bins persist in project storage.

In [ ]:
!python -m src.data.prepare --config configs/studiolab.yaml

## 2. Train (GPU runtime)
Auto-resumes from `results/ckpt.pt`. If a session ends before `max_iters`, just re-run this cell in a new GPU session. Add `--fresh` to restart from scratch.

In [ ]:
!python -m src.train --config configs/studiolab.yaml

## 3. Sample from the best checkpoint

In [ ]:
!python -m src.sample --config configs/studiolab.yaml --prompt "Once upon a time"

## 4. Push to the Hugging Face Hub
First, in a **Terminal**, log in with a **write** token (create one at https://huggingface.co/settings/tokens):
```bash
huggingface-cli login
```
Then push weights + tokenizer + an honest model card to `Abhilash-AI-Lab/saffron-v1` (the repo is created automatically):

In [ ]:
!python -m src.push_hf --config configs/sagemaker.yaml --repo Abhilash-AI-Lab/saffron-v1

## 5. Bring the weights back (optional)
`results/saffron.pt` is the best model. Once pushed to Hugging Face you can pull it anywhere, or download it via the Jupyter file browser (right-click → Download). **Do not** commit weights to GitHub (they're git-ignored).